In [ ]:
#import packages
import numpy as np
import netCDF4 as nc
import xarray as xr
import pandas as pd
import geopandas as gpd
from sklearn.linear_model import TheilSenRegressor
import pymannkendall as mk
import os
import gc

#add path and prepared data
path1=' '
in_path=path1+" "
out_path=path1+"results/"

som_grids = {'grid3x3': (3, 3), 'grid3x4': (3, 4),'grid4x4': (4, 4),}    

In [ ]:
gph_anom=xr.open_dataarray(in_path+"uyr_gph_anom.nc")
print(gph_anom)
time=gph_anom.time.values
lat=gph_anom.latitude.values
lon=gph_anom.longitude.values

shp = gpd.read_file("Upper_Yangtze_River.shp")

def clip_arr(arr):
    arr.rio.write_crs("epsg:4326", inplace=True)
    arr.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude", inplace=True)
    clip_data=arr.rio.clip(shp.geometry, shp.crs, drop=False, all_touched=True)
    return clip_data

clip_da=clip_arr(gph_anom)

#Record grid point coordinate locations
rigional_combo=[]
combo=[] 
da=clip_da[0,:,:]
for j in range(gph_anom.shape[1]):
    for k in range(gph_anom.shape[2]):
        combo.append((j,k))
        if not np.isnan(da[j, k].values):
            rigional_combo.append((j, k))           
#print(rigional_combo) #len=1674

indices = []
for item in rigional_combo:
    idx = combo.index(item)
    indices.append(idx)
print(indices)   #len=1674

gc.collect()

In [ ]:
#Seasonally separated indices facilitate subsequent processing

gph_anom['time'] = pd.to_datetime(gph_anom.time.values)

def get_season_indices(data_array):
    month = data_array.time.dt.month

    seasons = {
        'Spring': [3, 4, 5],
        'Summer': [6, 7, 8],
        'Autumn': [9, 10, 11],
        'Winter': [12, 1, 2]
    }
    
    season_ind = {}
    for season, months in seasons.items():
        
        mask = month.isin(months)
        
        season_ind[season] = np.where(mask)[0]
        
        print(f"{season}: {len(season_ind[season])} days")
        
    return season_ind

# get indices
season_ind = get_season_indices(gph_anom)

for season in season_ind:
    print(f"{season} : {season_ind[season]}")

gc.collect()

Spring: 4048 days
Summer: 4048 days
Autumn: 4004 days
Winter: 3960 days
Spring : [   59    60    61 ... 15843 15844 15845]
Summer : [  151   152   153 ... 15935 15936 15937]
Autumn : [  243   244   245 ... 16026 16027 16028]
Winter : [    0     1     2 ... 16057 16058 16059]


0

In [ ]:
# Calculate characteristics, trends and significance levels
# whole year
result={}
result_yearly={}
years = np.arange(1979, 2023)

# Theil-Sen
ts_model = TheilSenRegressor(fit_intercept=True, random_state=42, max_iter=500)

for grid_name, grid_size in som_grids.items():
    result[grid_name]={}
    result_yearly[grid_name]={}

    input_combo=[]  
    for j in range(grid_size[0]):
        for k in range(grid_size[1]):                 
                input_combo.append((j,k))     
  
    result[grid_name]=pd.DataFrame(np.nan, index=range(len(input_combo)), columns=['occur', 'slope_per', 'intercept', 'mk_result_per_z', 'mk_result_per_p'])
                                                                                    
    result_yearly[grid_name]={}

    classif=pd.read_csv(out_path+"gph_"+grid_name+"_total_classif.csv")  #pd.DataFrame
    #print(classif.shape)  (4048, 1)
    classif["time"]=time.reshape((len(time),1))
    classif["num"]=np.arange(1, len(time)+1, 1)
    group = classif.groupby('node').agg({'time': list, 'num': list})
    #print(group)

    for n in range(len(input_combo)):
        print(f'{grid_name}_{n}')
        result_yearly[grid_name][n]=pd.DataFrame(np.nan, index=years, columns=['yearly_counts', 'yearly_per'])
        node_data=group.loc[n+1]   #pandas.series
        node_df = pd.DataFrame({'time': node_data['time'], 'num': node_data['num']})   #pd.DataFrame
        
        node_df['year'] = node_df['time'].dt.year
        year_uniq=np.unique(node_df['year'])
        node_df['num_diff'] = node_df['num'].diff().fillna(0)

        #total occurrence
        node_occur=len(node_data['time'])  
        #print(node_occur)
        result[grid_name].loc[n, 'occur']=node_occur
    
        #fre yearly
        yearly_counts = node_df.groupby('year')['time'].count().values
        #print(yearly_counts)
        yearly_per=yearly_counts/np.sum(yearly_counts)
        #print("yearly_per", yearly_per)

        for i in range(len(year_uniq)):
            result_yearly[grid_name][n].loc[year_uniq[i], 'yearly_counts'] = yearly_counts[i]
            result_yearly[grid_name][n].loc[year_uniq[i], 'yearly_per'] = yearly_per[i]
    
        ts_model.fit(years.reshape(-1, 1), result_yearly[grid_name][n]['yearly_counts'].fillna(0))
        slope_per = ts_model.coef_[0]
        intercept = ts_model.intercept_
        mk_result_per = mk.original_test(result_yearly[grid_name][n]['yearly_counts'].fillna(0))
        result[grid_name].loc[n, 'slope_per']=slope_per
        result[grid_name].loc[n, 'intercept']=intercept
        result[grid_name].loc[n, 'mk_result_per_z']=mk_result_per.z
        result[grid_name].loc[n, 'mk_result_per_p']=mk_result_per.p

    dfs = result_yearly[grid_name]

    output_dir = out_path+'feature/'
    os.makedirs(output_dir, exist_ok=True)

    with pd.ExcelWriter(output_dir+'gph_node_yearly_' + grid_name + '_total.xlsx', engine='openpyxl') as writer:
        for key_name, df in dfs.items():
            df.to_excel(writer, sheet_name=f'node_{int(key_name) + 1}', index=False, header=True)

    pd.DataFrame(result[grid_name]).to_excel(output_dir+'gph_node_'+grid_name+'_total.xlsx', engine='openpyxl')    

    gc.collect()

In [ ]:
#same as above, but for seasonal
result={}
result_yearly={}
years = np.arange(1979, 2023)
# Theil-Sen
ts_model = TheilSenRegressor(fit_intercept=True, random_state=42, max_iter=500)

for grid_name, grid_size in som_grids.items():
    result[grid_name]={}
    result_yearly[grid_name]={}

    input_combo=[]  
    for j in range(grid_size[0]):
        for k in range(grid_size[1]):                 
                input_combo.append((j,k))   

    season_classif = {}

    classif=pd.read_csv(out_path+"gph_"+grid_name+"_total_classif.csv")  #pd.DataFrame
    #print(classif.shape)  (4048, 1)
    #classif.rename(columns={'x': 'node'}, inplace=True)
    classif["time"]=time.reshape((len(time),1))
    #classif['Time'] = pd.to_datetime(df['Time'])  
    classif["num"]=np.arange(1, len(time)+1, 1)

    for season, indices in season_ind.items():
        result[grid_name][season]=pd.DataFrame(np.nan, index=range(len(input_combo)), columns=['occur',  'slope_per', 'intercept', 'mk_result_per_z', 'mk_result_per_p'])
                                                                                                                                                
        result_yearly[grid_name][season]={}

        season_class = pd.DataFrame(columns=classif.columns)
        for index, row in classif.iterrows():
            node_num = np.array(row['num'])-1
            mask = np.isin(node_num, indices)
            if np.any(mask):
                new_row = {}
                for col in classif.columns:
                    if isinstance(row[col], list):
                        new_row[col] = np.array(row[col])[mask].tolist()
                    else:
                        new_row[col] = row[col]
                
                season_class.loc[index] = new_row
        
        # seasonal data
        season_classif[season] = season_class
        
        group = season_classif[season].groupby('node').agg({'time': list, 'num': list})

        for n in range(len(input_combo)):
            print(f'{grid_name}_{season}_{n}')
            result_yearly[grid_name][season][n]=pd.DataFrame(np.nan, index=years, columns=['yearly_counts', 'yearly_per'])
            if n+1 not in group.index:
                continue
            else:
                node_data=group.loc[n+1]   #pandas.series
            node_df = pd.DataFrame({'time': node_data['time'], 'num': node_data['num']})   #pd.DataFrame
            
            node_df['year'] = node_df['time'].dt.year
            year_uniq=np.unique(node_df['year'])
            #print("year_uniq:", year_uniq)
        
            node_df['num_diff'] = node_df['num'].diff().fillna(0)

            # total occur
            node_occur=len(node_data['time']) 
            #print(node_occur)
            result[grid_name][season].loc[n, 'occur']=node_occur

            #fre yearly
            yearly_counts = node_df.groupby('year')['time'].count().values
            #print(yearly_counts)
            yearly_per=yearly_counts/np.sum(yearly_counts)
            #print("yearly_per", yearly_per)

            num1=np.zeros(len(years))
            num2=np.zeros(len(years))
            for i in range(len(year_uniq)):
                num1[year_uniq[i]-1979]=yearly_counts[i]
                num2[year_uniq[i]-1979]=yearly_per[i]
            #print(num1)

            result_yearly[grid_name][season][n]['yearly_counts'] = num1
            result_yearly[grid_name][season][n]['yearly_per'] = num2
            
            ts_model.fit(years.reshape(-1, 1), result_yearly[grid_name][season][n]['yearly_counts'].fillna(0))
            slope_per = ts_model.coef_[0]
            #print(slope_per)
            intercept = ts_model.intercept_
            mk_result_per = mk.original_test(result_yearly[grid_name][season][n]['yearly_counts'].fillna(0))
            result[grid_name][season].loc[n, 'slope_per']=slope_per
            result[grid_name][season].loc[n, 'intercept']=intercept
            result[grid_name][season].loc[n, 'mk_result_per_z']=mk_result_per.z
            result[grid_name][season].loc[n, 'mk_result_per_p']=mk_result_per.p


        dfs = result_yearly[grid_name][season]
        with pd.ExcelWriter(output_dir+'gph_node_yearly_' + grid_name + '_' + season + '.xlsx', engine='openpyxl') as writer:
            for key_name, df in dfs.items():
                df.to_excel(writer, sheet_name=f'node_{int(key_name) + 1}', index=False, header=True)

        pd.DataFrame(result[grid_name][season]).to_excel(output_dir+'gph_node_'+grid_name+'_'+season+'.xlsx', engine='openpyxl')    